# LSH 스테이션 재고 예측 분석

이번 분석은 `정류장정보_시간대별_합친것.csv`를 이용해 다음 3개 시작 스테이션만 대상으로 진행한다.

- `ST-464`
- `ST-481`
- `ST-479`

목표는 두 가지다.

1. 사용자가 특정 시각에 3개 스테이션 중 한 곳을 방문했을 때 자전거를 탈 수 있을지 예측한다.
2. 운영자가 오전 4시와 오후 4시에 몇 대를 채워 넣어야 하는지 계산한다.


## 분석 가정과 방법

데이터에는 실시간 잔여 자전거 대수가 아니라 시간대별 이동 기록이 들어 있다. 따라서 아래와 같은 운영 규칙을 두고 재고를 시뮬레이션한다.

- `출발시간` 집계는 해당 스테이션 재고를 감소시키는 이벤트로 본다.
- `도착시간` 집계는 해당 스테이션 재고를 증가시키는 이벤트로 본다.
- 재고 보충은 매일 `04:00`와 `16:00` 두 번만 한다.
- 각 12시간 구간 안에서 재고가 음수가 되지 않도록 필요한 최소 시작 재고를 `목표 재고(target inventory)`로 둔다.
- 실제 보충 대수는 `현재 남은 재고`와 `목표 재고`의 차이만큼만 채운다.

즉, 이 노트북은 다음 질문에 답한다.

- `몇 시에 방문하면 탈 수 있을까?` -> 해당 시각 직전까지의 누적 재고가 1대 이상일 확률로 본다.
- `채우는 사람은 몇 대를 넣어야 하나?` -> 04시와 16시에 실제로 추가 투입해야 하는 평균 대수로 본다.

주의할 점은, 이 결과가 실시간 IoT 재고 데이터 기반 예측은 아니라는 것이다. 대신 2024년 이동 패턴을 기준으로 한 `운영 시뮬레이션형 예측`이다.


In [4]:
import csv
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path
from statistics import mean

TARGET_STATIONS = ["ST-464", "ST-481", "ST-479"]
VISIT_HOUR = 8  # 원하는 방문 시각으로 바꿔서 다시 실행하면 된다.

CANDIDATE_PATHS = [
    Path("Data/정류장정보_시간대별_합친것.csv"),
    Path("../../Data/정류장정보_시간대별_합친것.csv"),
]

for candidate in CANDIDATE_PATHS:
    if candidate.exists():
        DATA_PATH = candidate
        break
else:
    raise FileNotFoundError("정류장정보_시간대별_합친것.csv 파일을 찾지 못했습니다.")


def parse_datetime(date_text, hhmm_text):
    dt = datetime.strptime(date_text, "%Y%m%d")
    hhmm = int(float(hhmm_text))
    return dt.replace(hour=hhmm // 100, minute=hhmm % 100)


def get_cycle_start(dt):
    if dt.hour < 4:
        previous_day = dt.date() - timedelta(days=1)
        return datetime.combine(previous_day, datetime.min.time()).replace(hour=16)
    if dt.hour < 16:
        return datetime.combine(dt.date(), datetime.min.time()).replace(hour=4)
    return datetime.combine(dt.date(), datetime.min.time()).replace(hour=16)


def load_station_events(data_path, target_stations):
    station_names = {}
    events = []

    with data_path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            trip_count = int(round(float(row["전체_건수"] or 0)))
            if trip_count == 0:
                continue

            event_time = parse_datetime(row["기준_날짜"], row["기준_시간대"])
            basis = row["집계_기준"]

            if basis == "출발시간" and row["시작_대여소_ID"] in target_stations:
                station_id = row["시작_대여소_ID"]
                station_names.setdefault(station_id, row["시작_대여소명"])
                events.append((station_id, event_time, -trip_count))

            if basis == "도착시간" and row["종료_대여소_ID"] in target_stations:
                station_id = row["종료_대여소_ID"]
                station_names.setdefault(station_id, row["종료_대여소명"])
                events.append((station_id, event_time, trip_count))

    return station_names, events


def build_cycle_events(events):
    cycle_events = defaultdict(list)
    for station_id, event_time, delta in events:
        cycle_events[(station_id, get_cycle_start(event_time))].append((event_time, delta))

    for key in cycle_events:
        cycle_events[key].sort()

    return cycle_events


def compute_target_inventory(cycle_events):
    targets = defaultdict(dict)
    for (station_id, cycle_start), items in cycle_events.items():
        cumulative = 0
        min_cumulative = 0
        for _, delta in items:
            cumulative += delta
            min_cumulative = min(min_cumulative, cumulative)
        targets[station_id][cycle_start] = -min_cumulative
    return targets


def inventory_at_time(start_inventory, items, visit_time):
    inventory = start_inventory
    for event_time, delta in items:
        if event_time < visit_time:
            inventory += delta
        else:
            break
    return inventory


def close_cycle(start_inventory, items):
    inventory = start_inventory
    for _, delta in items:
        inventory += delta
        if inventory < 0:
            inventory = 0
    return inventory


def run_simulation(target_stations, visit_hour):
    station_names, events = load_station_events(DATA_PATH, target_stations)
    cycle_events = build_cycle_events(events)
    cycle_targets = compute_target_inventory(cycle_events)
    all_cycle_starts = sorted({cycle_start for _, cycle_start in cycle_events})

    station_results = {}
    for station_id in target_stations:
        inventory = 0
        rows = []
        for cycle_start in all_cycle_starts:
            items = cycle_events.get((station_id, cycle_start), [])
            target_inventory = cycle_targets[station_id].get(cycle_start, 0)
            add_bikes = max(0, target_inventory - inventory)
            inventory += add_bikes

            visit_time = cycle_start + timedelta(hours=(visit_hour - cycle_start.hour) % 24)
            if visit_time >= cycle_start + timedelta(hours=12):
                visit_inventory = None
                available = None
            else:
                visit_inventory = inventory_at_time(inventory, items, visit_time)
                available = visit_inventory > 0

            end_inventory = close_cycle(inventory, items)
            rows.append(
                {
                    "cycle_start": cycle_start,
                    "target_inventory": target_inventory,
                    "add_bikes": add_bikes,
                    "start_inventory": inventory,
                    "items": items,
                    "visit_inventory": visit_inventory,
                    "available": available,
                    "end_inventory": end_inventory,
                }
            )
            inventory = end_inventory
        station_results[station_id] = rows

    return station_names, station_results


def availability_probability_for_hour(rows, visit_hour):
    results = []
    for row in rows:
        cycle_start = row["cycle_start"]
        visit_time = cycle_start + timedelta(hours=(visit_hour - cycle_start.hour) % 24)
        if visit_time >= cycle_start + timedelta(hours=12):
            continue
        inventory = inventory_at_time(row["start_inventory"], row["items"], visit_time)
        results.append(1 if inventory > 0 else 0)
    return round(mean(results), 3)


def summarize_results(station_names, station_results, visit_hour):
    summary_rows = []
    hourly_rows = []

    for station_id, rows in station_results.items():
        visit_probability = availability_probability_for_hour(rows, visit_hour)
        avg_fill_4 = mean(row["add_bikes"] for row in rows if row["cycle_start"].hour == 4)
        avg_fill_16 = mean(row["add_bikes"] for row in rows if row["cycle_start"].hour == 16)
        avg_fill_total = mean(row["add_bikes"] for row in rows)

        summary_rows.append(
            {
                "station_id": station_id,
                "station_name": station_names.get(station_id, ""),
                "visit_hour": visit_hour,
                "availability_probability": round(visit_probability, 3),
                "avg_fill_4": round(avg_fill_4, 2),
                "avg_fill_16": round(avg_fill_16, 2),
                "avg_fill_total": round(avg_fill_total, 2),
            }
        )

    for station_id, rows in station_results.items():
        for hour in range(24):
            availability_probability = availability_probability_for_hour(rows, hour)
            hourly_rows.append(
                {
                    "station_id": station_id,
                    "station_name": station_names.get(station_id, ""),
                    "visit_hour": hour,
                    "availability_probability": round(availability_probability, 3),
                }
            )

    return summary_rows, hourly_rows


def format_table(rows, columns):
    display_rows = [[str(row[column]) for column in columns] for row in rows]
    widths = []
    for index, column in enumerate(columns):
        max_width = max(len(column), *(len(item[index]) for item in display_rows))
        widths.append(max_width)

    header = " | ".join(column.ljust(widths[index]) for index, column in enumerate(columns))
    divider = "-+-".join("-" * widths[index] for index in range(len(columns)))
    body = [" | ".join(item[index].ljust(widths[index]) for index in range(len(columns))) for item in display_rows]
    return "\n".join([header, divider, *body])


station_names, station_results = run_simulation(TARGET_STATIONS, VISIT_HOUR)
summary_rows, hourly_rows = summarize_results(station_names, station_results, VISIT_HOUR)


In [5]:
print(f"데이터 경로: {DATA_PATH}")
print(f"기준 방문 시각: {VISIT_HOUR:02d}:00")
print()
print("[방문 가능성과 보충 대수 요약]")
print(
    format_table(
        sorted(summary_rows, key=lambda row: (-row["availability_probability"], row["station_id"])),
        [
            "station_id",
            "station_name",
            "visit_hour",
            "availability_probability",
            "avg_fill_4",
            "avg_fill_16",
            "avg_fill_total",
        ],
    )
)

print()
print("[시간대별 방문 가능 확률]")
for station_id in TARGET_STATIONS:
    station_hour_rows = [row for row in hourly_rows if row["station_id"] == station_id]
    print()
    print(f"{station_id} / {station_names.get(station_id, '')}")
    print(
        format_table(
            station_hour_rows,
            ["visit_hour", "availability_probability"],
        )
    )


데이터 경로: ../../Data/정류장정보_시간대별_합친것.csv
기준 방문 시각: 08:00

[방문 가능성과 보충 대수 요약]
station_id | station_name | visit_hour | availability_probability | avg_fill_4 | avg_fill_16 | avg_fill_total
-----------+--------------+------------+--------------------------+------------+-------------+---------------
ST-481     | 구산동_062_1    | 8          | 1                        | 1.7        | 0.58        | 1.14          
ST-464     | 역촌동_022_1    | 8          | 0.994                    | 1.84       | 1.35        | 1.59          
ST-479     | 역촌동_068_1    | 8          | 0.961                    | 2.46       | 0.76        | 1.6           

[시간대별 방문 가능 확률]

ST-464 / 역촌동_022_1
visit_hour | availability_probability
-----------+-------------------------
0          | 0.992                   
1          | 0.981                   
2          | 0.975                   
3          | 0.956                   
4          | 0.992                   
5          | 0.989                   
6          | 0.

## 해석 방법

- `availability_probability`는 해당 시각에 방문했을 때 자전거가 1대 이상 남아 있을 확률이다.
- `avg_fill_4`는 오전 4시에 평균적으로 추가 투입해야 하는 자전거 대수다.
- `avg_fill_16`는 오후 4시에 평균적으로 추가 투입해야 하는 자전거 대수다.
- `avg_fill_total`은 두 주기를 합쳐 평균적으로 한 번의 보충 시점에 넣어야 하는 대수다.

예를 들어 `availability_probability`가 `0.97`이면, 과거 패턴 기준으로 같은 운영 규칙을 적용했을 때 약 97% 확률로 탑승 가능한 것으로 해석한다.


In [6]:
best_station = max(summary_rows, key=lambda row: row["availability_probability"])
highest_fill_4 = max(summary_rows, key=lambda row: row["avg_fill_4"])
highest_fill_16 = max(summary_rows, key=lambda row: row["avg_fill_16"])

print(f"{VISIT_HOUR:02d}:00 방문 기준 가장 안정적인 스테이션은 {best_station['station_id']} ({best_station['station_name']}) 입니다.")
print(f"예상 탑승 가능 확률은 {best_station['availability_probability']:.1%} 입니다.")
print()
print(f"오전 4시 기준 가장 많이 채워야 하는 곳은 {highest_fill_4['station_id']} 이고 평균 {highest_fill_4['avg_fill_4']}대 입니다.")
print(f"오후 4시 기준 가장 많이 채워야 하는 곳은 {highest_fill_16['station_id']} 이고 평균 {highest_fill_16['avg_fill_16']}대 입니다.")

print()
print("발표나 보고서에는 아래 문장 형태로 바로 옮겨 적으면 된다.")
print(
    f"- {VISIT_HOUR:02d}시에 방문한다고 가정하면, 세 스테이션 중 {best_station['station_id']}의 탑승 가능성이 가장 높고 "
    f"예상 확률은 약 {best_station['availability_probability']:.1%}이다."
)
print(
    f"- 운영 측면에서는 오전 4시에는 {highest_fill_4['station_id']}에 평균 {highest_fill_4['avg_fill_4']}대, "
    f"오후 4시에는 {highest_fill_16['station_id']}에 평균 {highest_fill_16['avg_fill_16']}대를 우선 배치하는 전략이 유효하다."
)


08:00 방문 기준 가장 안정적인 스테이션은 ST-481 (구산동_062_1) 입니다.
예상 탑승 가능 확률은 100.0% 입니다.

오전 4시 기준 가장 많이 채워야 하는 곳은 ST-479 이고 평균 2.46대 입니다.
오후 4시 기준 가장 많이 채워야 하는 곳은 ST-464 이고 평균 1.35대 입니다.

발표나 보고서에는 아래 문장 형태로 바로 옮겨 적으면 된다.
- 08시에 방문한다고 가정하면, 세 스테이션 중 ST-481의 탑승 가능성이 가장 높고 예상 확률은 약 100.0%이다.
- 운영 측면에서는 오전 4시에는 ST-479에 평균 2.46대, 오후 4시에는 ST-464에 평균 1.35대를 우선 배치하는 전략이 유효하다.


## MAE 평가

이번 셀에서는 `MAE(Mean Absolute Error)`를 계산한다.

`MAE`는 예측값과 실제값의 차이를 절댓값으로 바꾼 뒤 평균낸 값이다.

$$
MAE = \\frac{1}{n} \\sum_{i=1}^{n} |y_i - \\hat{y}_i|
$$

여기서는 다음처럼 정의한다.

- 실제값 `y_i`: 각 12시간 주기에서 실제로 필요했던 보충 대수(`add_bikes`)
- 예측값 `\\hat{y}_i`: 같은 스테이션, 같은 보충 시점(04시 또는 16시)의 평균 보충 대수

즉 이 평가는 `평균적으로 운영하면 실제 필요 대수에서 몇 대 정도 차이 나는가`를 보여준다.

- `MAE`가 작을수록 평균 전략이 안정적이다.
- `MAE`가 크면 날짜별 편차가 크므로, 요일/월/날씨 같은 추가 변수를 넣은 모델이 더 필요하다는 뜻이다.

같은 셀에서 `RMSE`, `R²`도 함께 계산한다. 여기서 `R²`는 평균값을 예측으로 쓰는 단순 기준선이라 보통 `0`에 가깝게 나온다.


In [7]:
import math
from collections import defaultdict
from html import escape

try:
    from IPython.display import HTML, display
except ModuleNotFoundError:
    HTML = lambda value: value
    def display(value):
        print("[SVG output generated]")
        print(str(value)[:200] + "...")


def mean_absolute_error(y_true, y_pred):
    return sum(abs(actual - predicted) for actual, predicted in zip(y_true, y_pred)) / len(y_true)


def root_mean_squared_error(y_true, y_pred):
    mse = sum((actual - predicted) ** 2 for actual, predicted in zip(y_true, y_pred)) / len(y_true)
    return math.sqrt(mse)


def r2_score(y_true, y_pred):
    y_mean = sum(y_true) / len(y_true)
    ss_res = sum((actual - predicted) ** 2 for actual, predicted in zip(y_true, y_pred))
    ss_tot = sum((actual - y_mean) ** 2 for actual in y_true)
    if ss_tot == 0:
        return 1.0
    return 1 - (ss_res / ss_tot)


slot_labels = {4: "04:00 refill", 16: "16:00 refill"}
evaluation_rows = []
plot_rows = []

for station_id, rows in station_results.items():
    grouped = defaultdict(list)
    for row in rows:
        grouped[row["cycle_start"].hour].append(row)

    for cycle_hour, slot_rows in sorted(grouped.items()):
        actual_values = [row["add_bikes"] for row in slot_rows]
        baseline_prediction = mean(actual_values)
        predicted_values = [baseline_prediction] * len(actual_values)

        mae_value = mean_absolute_error(actual_values, predicted_values)
        rmse_value = root_mean_squared_error(actual_values, predicted_values)
        r2_value = r2_score(actual_values, predicted_values)

        evaluation_rows.append(
            {
                "station_id": station_id,
                "station_name": station_names.get(station_id, ""),
                "slot": slot_labels.get(cycle_hour, str(cycle_hour)),
                "predicted_fill": round(baseline_prediction, 2),
                "mae": round(mae_value, 2),
                "rmse": round(rmse_value, 2),
                "r2": round(r2_value, 3),
            }
        )

        for actual_value in actual_values:
            plot_rows.append(
                {
                    "station_id": station_id,
                    "slot": slot_labels.get(cycle_hour, str(cycle_hour)),
                    "actual": actual_value,
                    "predicted": baseline_prediction,
                }
            )

print("[MAE / RMSE / R2 평가]")
print(
    format_table(
        evaluation_rows,
        ["station_id", "station_name", "slot", "predicted_fill", "mae", "rmse", "r2"],
    )
)

print()
print("해석:")
print("- predicted_fill: 같은 스테이션과 같은 보충 시점에서 평균적으로 넣게 되는 대수")
print("- mae: 그 평균 전략이 실제 필요 대수와 평균적으로 몇 대 차이 나는지")
print("- rmse: 큰 오차에 더 민감한 지표")
print("- r2: 평균값을 그대로 예측으로 사용한 기준선이라 대체로 0에 가깝다")

def svg_scatter_panels(evaluation_rows, plot_rows):
    panel_width = 420
    panel_height = 260
    cols = 2
    rows_count = math.ceil(len(evaluation_rows) / cols)
    total_width = panel_width * cols
    total_height = panel_height * rows_count
    pieces = [f'<svg width="{total_width}" height="{total_height}" xmlns="http://www.w3.org/2000/svg">']

    for index, row in enumerate(evaluation_rows):
        col = index % cols
        row_idx = index // cols
        x0 = col * panel_width
        y0 = row_idx * panel_height
        subset = [item for item in plot_rows if item["station_id"] == row["station_id"] and item["slot"] == row["slot"]]
        actual_values = [item["actual"] for item in subset]
        predicted_values = [item["predicted"] for item in subset]
        upper = max(max(actual_values), max(predicted_values), 1)
        left = x0 + 55
        right = x0 + panel_width - 20
        top = y0 + 45
        bottom = y0 + panel_height - 45

        pieces.append(f'<rect x="{x0+8}" y="{y0+8}" width="{panel_width-16}" height="{panel_height-16}" fill="white" stroke="#d0d7de"/>')
        pieces.append(f'<text x="{x0+20}" y="{y0+25}" font-size="14" font-weight="bold">{escape(row["station_id"])} | {escape(row["slot"])} </text>')
        pieces.append(f'<text x="{x0+20}" y="{y0+40}" font-size="12">MAE={row["mae"]}, R2={row["r2"]}</text>')
        pieces.append(f'<line x1="{left}" y1="{bottom}" x2="{right}" y2="{bottom}" stroke="#333"/>')
        pieces.append(f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#333"/>')
        pieces.append(f'<line x1="{left}" y1="{bottom}" x2="{right}" y2="{top}" stroke="#d62728" stroke-dasharray="5,5"/>')
        pieces.append(f'<text x="{(left+right)/2}" y="{y0+panel_height-10}" text-anchor="middle" font-size="12">Actual refill bikes</text>')
        pieces.append(f'<text x="{x0+14}" y="{(top+bottom)/2}" font-size="12" transform="rotate(-90 {x0+14} {(top+bottom)/2})">Predicted refill bikes</text>')
        pieces.append(f'<text x="{left}" y="{bottom+16}" font-size="10">0</text>')
        pieces.append(f'<text x="{right-10}" y="{bottom+16}" font-size="10">{upper:.1f}</text>')
        pieces.append(f'<text x="{left-22}" y="{bottom}" font-size="10">0</text>')
        pieces.append(f'<text x="{left-30}" y="{top+4}" font-size="10">{upper:.1f}</text>')

        for actual, predicted in zip(actual_values, predicted_values):
            cx = left + (actual / upper) * (right - left)
            cy = bottom - (predicted / upper) * (bottom - top)
            pieces.append(f'<circle cx="{cx:.2f}" cy="{cy:.2f}" r="3" fill="#1f77b4" fill-opacity="0.55"/>')

    pieces.append('</svg>')
    return ''.join(pieces)


def svg_bar_chart(evaluation_rows):
    width = 980
    height = 360
    left = 70
    right = width - 30
    top = 40
    bottom = height - 90
    max_value = max(row["mae"] for row in evaluation_rows) or 1
    colors = ["#4e79a7", "#a0cbe8", "#59a14f", "#8cd17d", "#e15759", "#ff9d9a"]
    slot_width = (right - left) / len(evaluation_rows)
    pieces = [f'<svg width="{width}" height="{height}" xmlns="http://www.w3.org/2000/svg">']
    pieces.append(f'<rect x="1" y="1" width="{width-2}" height="{height-2}" fill="white" stroke="#d0d7de"/>')
    pieces.append(f'<text x="{width/2}" y="24" text-anchor="middle" font-size="16" font-weight="bold">MAE of Refill Prediction by Station and Refill Slot</text>')
    pieces.append(f'<line x1="{left}" y1="{bottom}" x2="{right}" y2="{bottom}" stroke="#333"/>')
    pieces.append(f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#333"/>')
    pieces.append(f'<text x="20" y="{(top+bottom)/2}" font-size="12" transform="rotate(-90 20 {(top+bottom)/2})">MAE (bikes)</text>')

    for idx, row in enumerate(evaluation_rows):
        bar_left = left + idx * slot_width + 20
        bar_width = slot_width - 40
        bar_height = (row["mae"] / max_value) * (bottom - top)
        bar_top = bottom - bar_height
        label = f"{row['station_id']} {row['slot']}"
        pieces.append(f'<rect x="{bar_left:.2f}" y="{bar_top:.2f}" width="{bar_width:.2f}" height="{bar_height:.2f}" fill="{colors[idx % len(colors)]}"/>')
        pieces.append(f'<text x="{bar_left + bar_width/2:.2f}" y="{bar_top - 6:.2f}" text-anchor="middle" font-size="11">{row["mae"]:.2f}</text>')
        pieces.append(f'<text x="{bar_left + bar_width/2:.2f}" y="{bottom + 18}" text-anchor="middle" font-size="10">{escape(row["station_id"])}</text>')
        pieces.append(f'<text x="{bar_left + bar_width/2:.2f}" y="{bottom + 32}" text-anchor="middle" font-size="10">{escape(row["slot"])}</text>')

    pieces.append('</svg>')
    return ''.join(pieces)


display(HTML(svg_scatter_panels(evaluation_rows, plot_rows)))
display(HTML(svg_bar_chart(evaluation_rows)))


[MAE / RMSE / R2 평가]
station_id | station_name | slot         | predicted_fill | mae  | rmse | r2 
-----------+--------------+--------------+----------------+------+------+----
ST-464     | 역촌동_022_1    | 04:00 refill | 1.84           | 2.27 | 3.06 | 0.0
ST-464     | 역촌동_022_1    | 16:00 refill | 1.35           | 1.71 | 2.31 | 0.0
ST-481     | 구산동_062_1    | 04:00 refill | 1.7            | 2.62 | 3.84 | 0.0
ST-481     | 구산동_062_1    | 16:00 refill | 0.58           | 0.96 | 1.75 | 0.0
ST-479     | 역촌동_068_1    | 04:00 refill | 2.46           | 3.78 | 5.43 | 0.0
ST-479     | 역촌동_068_1    | 16:00 refill | 0.76           | 1.26 | 2.38 | 0.0

해석:
- predicted_fill: 같은 스테이션과 같은 보충 시점에서 평균적으로 넣게 되는 대수
- mae: 그 평균 전략이 실제 필요 대수와 평균적으로 몇 대 차이 나는지
- rmse: 큰 오차에 더 민감한 지표
- r2: 평균값을 그대로 예측으로 사용한 기준선이라 대체로 0에 가깝다
